# Rapport MS COCO classification challenge

To run this notebook successfully, you need to have the MS COCO dataset downloaded and every python package installed. All python requirements are listed in the appropriate files given in the zip file and was created using uv. You can also create an.env file to provide different variables to the notebook using .env.example as a template. The notebook is structured in several sections, each of them being a code cell or a Markdown cell. You can run the notebook cell by cell, or you can run all cells at once.

## 1. Installation and imports

We start by importing all our libraries needed for this notebook. We also check the version of Python and PyTorch, and we initialize the device to be used (CPU or GPU). Setting DEVICE in the .env is not compulsory as this code will try to detect the best device available. If you want to force the use of a specific device, you can set the DEVICE variable in the .env file to `cpu`, `cuda`, or `mps` for example.

In [ ]:
# import statements for python, torch and companion libraries and your own modules
import torch
import os
import platform
import numpy as np
import random
import matplotlib.pyplot as plt
import pandas as pd

from dotenv import load_dotenv
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score

from lib.util import CLASSES, COCOTrainImageDataset, COCOTestImageDataset, train_loop, validation_loop, update_graphs

load_dotenv()

# device initialization
device_name = os.getenv("DEVICE", "")
DEVICE = torch.device(
    device_name) if device_name else (
    torch.accelerator.current_accelerator()) if (
    torch.accelerator.is_available()) else (
    torch.device("cpu"))

print("Python :", platform.python_version())
print("PyTorch :", torch.__version__)
print("Device used :", DEVICE)

Potentiel import supp à supprimer plus tard

In [ ]:
import json, time, copy, platform, math
from collections import Counter
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models



## 2. Configuration

In this section, we will define the configuration parameters for our experiment. We start by defining the paths to the data and output directories. We strongly advise defining an appropriate DATA_ROOT corresponding to the path to the ms coco dataset either by creating a .env file or directly by replacing the default value set below. We also define some hyperparameters for training, such as the number of epochs, batch size, and learning rate. We also set a random seed for reproducibility and define a quick run option to limit the number of images processed for testing purposes. Before continuing, verify that all the output paths are correct and that the dataset is accessible.

In [ ]:
# Defining global variables for data directories
DATA_ROOT = Path(os.getenv("DATA_ROOT", "/opt/iav/labs/data"))
DATA_ROOT = DATA_ROOT / "ms-coco"
IMAGE_DIR = DATA_ROOT / "images/train"
LABEL_DIR = DATA_ROOT / "labels/train"
TEST_IMAGE_DIR = DATA_ROOT / "images/test"

# Defining global variables for output directories
OUTPUT_DIR = Path(os.getenv("OUTPUT_DIR", "output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_PATH = OUTPUT_DIR / os.getenv("JSON_PATH", "submission.json")

# TODO : check if all variables are defined
SEED = 30
EPOCHS = 10
BATCH_SIZE = 32
NUM_WORKERS = min(8, os.cpu_count() or 1)
IMAGE_SIZE = 224
PATIENCE = 3
VAL_FRACTION = 0.15  # Fraction of the dataset to be used for validation
THRESHOLD = 0.5  # Probability threshold above which a class is predicted

TRANSFORM = False

# For a quick run, you can set QUICK_RUN to True and limit the number of images processed
QUICK_RUN = False
QUICK_MAX_IMAGES = 5000

# Pin memory for faster data transfer to GPU if using CUDA
PIN_MEMORY = DEVICE.type == "cuda"


# Function to set random seed for reproducibility
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = (DEVICE.type == "cuda")


set_seed()

print("Images :", IMAGE_DIR.resolve())
print("Annotations :", LABEL_DIR.resolve())
print("Test images :", TEST_IMAGE_DIR.resolve())
print("Outputs :", OUTPUT_DIR.resolve())
print("Submission :", JSON_PATH.resolve())

## 3. Classes definitions and files verification

Here we define the classes and verify that we can access the images and annotations. If everything works fine, you should see the number of images and annotations found, and they should be equal to each other (Normally, there is 65 000 images if the dataset is correctly downloaded). You should also see an example of an image and its corresponding annotation file.

In [ ]:
classes = CLASSES
NUM_CLASSES = len(classes)
assert NUM_CLASSES == 80, f"We expect to find 80 classes but {NUM_CLASSES} were found."

if not IMAGE_DIR.is_dir() or not LABEL_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset not found. Verify DATA_ROOT={DATA_ROOT.resolve()} ; "
        f"There must be a folder 'images/train' and a folder 'labels/train' containing the .jpg images and .cls annotation files respectively."
    )

test_images = sorted(IMAGE_DIR.glob("*.jpg"))
label_files = sorted(LABEL_DIR.glob("*.cls"))

if not test_images:
    raise RuntimeError("No .jpg files found in IMAGE_DIR.")
if not label_files:
    raise RuntimeError("No .cls files found in LABEL_DIR.")

print(f"Images : {len(test_images):,}")
print(f"Annotated files : {len(label_files):,}")
print("Image example :", test_images[0].name if test_images else None)
print("annotation example :", label_files[0].name if label_files else None)

if len(test_images) != len(label_files):
    # Warning if the number of images and labels do not match
    print(
        f"Warning: Number of images ({len(test_images)}) does not match number of labels ({len(label_files)}). Some images may not have corresponding labels or vice versa.")

## 4. Annotations analysis

Before going straight to the training part, we will analyze the annotations to see the distribution of classes before any training in order to analyse the composition of the dataset.

In [ ]:
scan_files = label_files[:QUICK_MAX_IMAGES] if QUICK_RUN else label_files
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
labels_per_image = []
invalid_rows = []

for p in tqdm(scan_files, desc="Annotations analysis"):
    try:
        ids = [int(x.strip()) for x in p.read_text().splitlines() if x.strip()]
        if any(i < 0 or i >= NUM_CLASSES for i in ids):
            invalid_rows.append(p.name)
            continue
        class_counts[np.unique(ids)] += 1
        labels_per_image.append(len(set(ids)))
    except Exception as e:
        print(f"Error processing {p.name}: {e}")
        invalid_rows.append(p.name)

print("Images analysed :", len(scan_files))
print("Annotations problematic :", len(invalid_rows))
print("Average number of classes/image :", round(float(np.mean(labels_per_image)), 3))
print("Median :", float(np.median(labels_per_image)))
print("Min : ", min(labels_per_image))
print("Max : ", max(labels_per_image))

# Displaying the distribution of images per class in a DataFrame
freq_df = pd.DataFrame({"class": classes, "images_positives": class_counts})
display(freq_df.sort_values("images_positives"))

# Distribution of the number of images per class
plt.figure(figsize=(16, 4))
plt.bar(np.arange(NUM_CLASSES), class_counts)
plt.title("Number of positive images per class")
plt.xlabel("Class identifier")
plt.ylabel("Number of images")
plt.tight_layout()
plt.show()

# Distribution of the number of classes per image
plt.figure(figsize=(7, 4))
plt.hist(labels_per_image, bins=range(1, max(labels_per_image) + 2), align="left")
plt.title("Number of classes per image")
plt.xlabel("Number of classes")
plt.ylabel("Images")
plt.tight_layout()
plt.show()

As we can see, the distribution of classes is highly imbalanced. Firstly, the class "person" is the most represented class by far with 37,000 images. This means that the dataset is heavily biased towards images containing people, which could potentially lead to biased model predictions if not addressed properly. We also have classes with very few images, such as `hair drier` or `toaster` with only roughly 100 images. This extreme imbalance can make it difficult for the model to learn to recognize these underrepresented classes, leading to poor performance on them. The average number of classes per image is around three, which indicates that most images contain multiple objects without overflowing with too many classes. The median number of classes per image is two, which means that half of the images contain two or fewer classes going in the direction of a low number of classes per image.

## 5. Dataset PyTorch

The PyTorch Dataset classes for training and testing are defined in `lib/util.py` and imported at the beginning of this notebook. These classes will handle loading the images and their corresponding labels, applying any necessary transformations, and preparing the data for training and evaluation. They are based on the provided function given in the starting notebook.

## 6. Transformations and DataLoaders

Here we define the transformations to be applied to the images if set, and we create the training and validation datasets and dataloaders.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

resize_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

max_images = QUICK_MAX_IMAGES if QUICK_RUN else None
base_dataset = COCOTrainImageDataset(IMAGE_DIR, LABEL_DIR, transform=None,
                                     maximum_images=max_images) if TRANSFORM else COCOTrainImageDataset(IMAGE_DIR,
                                                                                                        LABEL_DIR,
                                                                                                        transform=resize_transform,
                                                                                                        maximum_images=max_images)
n_val = max(1, int(len(base_dataset) * VAL_FRACTION))
n_train = len(base_dataset) - n_val
generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = torch.utils.data.random_split(base_dataset, [n_train, n_val], generator=generator)


class TransformSubset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset, self.transform = subset, transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        original_idx = self.subset.indices[index]
        label_path = self.subset.dataset.label_files[original_idx]
        image = Image.open(self.subset.dataset.img_dir / f"{label_path.stem}.jpg").convert("RGB")
        labels = [int(x.strip()) for x in label_path.read_text().splitlines() if x.strip()]
        labels = torch.zeros(80).scatter_(0, torch.tensor(labels), value=1)
        return self.transform(image), labels


train_ds = TransformSubset(train_subset, train_transform) if TRANSFORM else train_subset
val_ds = TransformSubset(val_subset, val_transform) if TRANSFORM else val_subset

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                        persistent_workers=NUM_WORKERS > 0)

print(f"Train : {len(train_ds):,} | Validation : {len(val_ds):,}")

## 7. Metrics computation

In [10]:
def compute_metrics(y_true, y_prob, threshold=THRESHOLD):
    y_pred = (y_prob >= threshold).astype(np.uint8)
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }


def compute_class_metrics(class_prec, class_recall):
    class_results = []
    for p, r in zip(class_prec, class_recall):
        f1 = (0 if p == r == 0 else 2. / (1 / p + 1 / r))
        class_results.append({"f1": f1, "precision": p, "recall": r})
    return class_results


@torch.no_grad()
def predict_loader(model, loader, device=DEVICE):
    model.eval()
    probs_all, targets_all = [], []
    for images, targets in tqdm(loader, leave=False):
        logits = model(images.to(device, non_blocking=True))
        probs_all.append(torch.sigmoid(logits).cpu().numpy())
        targets_all.append(targets.numpy())
    return np.concatenate(targets_all), np.concatenate(probs_all)


def inverse_frequency_weighted_f1(y_true, y_pred):
    # Approximation explicite de la logique fournie dans le notebook de cours.
    positives = y_true.sum(axis=0)
    tp = ((y_true == 1) & (y_pred == 1)).sum(axis=0)
    fp = ((y_true == 0) & (y_pred == 1)).sum(axis=0)
    precision = np.divide(tp, tp + fp, out=np.zeros_like(tp, dtype=float), where=(tp + fp) > 0)
    recall = np.divide(tp, positives, out=np.zeros_like(tp, dtype=float), where=positives > 0)
    weights = np.divide(1.0, positives, out=np.zeros_like(positives, dtype=float), where=positives > 0)
    if weights.sum() == 0: return 0.0
    weights = weights / weights.sum()
    p, r = np.sum(precision * weights), np.sum(recall * weights)
    return 0.0 if p + r == 0 else 2 * p * r / (p + r)


def threshold_sweep(y_true, y_prob, thresholds=np.arange(0.05, 0.96, 0.05)):
    rows = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(np.uint8)
        rows.append({
            "threshold": round(float(t), 2),
            "f1_leaderboard": inverse_frequency_weighted_f1(y_true, y_pred),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        })
    sweep_df = pd.DataFrame(rows)
    best = sweep_df.loc[sweep_df["f1_leaderboard"].idxmax()]
    return sweep_df, best

## 8. Pretrained models and training functions

Here, we initialize our pretrained models: ResNet18, ResNet50 and EfficientNet-B0. Since these models have already been trained on ImageNet, we replace their classification head with a new 80-output layer and either fine-tune the whole network or freeze every layer except this new head (`fc` for ResNets, `classifier` for EfficientNet).

We decided to use pretrained model, because these model are already trained on huge dataset who well represent objects in our common life. By this way, we're convised that result will be beter. 

In [11]:
def build_model(backbone="resnet18", pretrained=True, freeze_backbone=False):
    if backbone == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        model = models.resnet18(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, NUM_CLASSES)
        if freeze_backbone:
            for name, p in model.named_parameters():
                if not name.startswith("fc."): p.requires_grad = False
    elif backbone == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, NUM_CLASSES)
        if freeze_backbone:
            for name, p in model.named_parameters():
                if not name.startswith("fc."): p.requires_grad = False
    elif backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
        if freeze_backbone:
            for name, p in model.named_parameters():
                if not name.startswith("classifier."): p.requires_grad = False
    else:
        raise ValueError(f"Backbone non pris en charge : {backbone}")
    return model.to(DEVICE)


def make_criterion(loss_name, train_targets=None):
    if loss_name == "bce":
        return nn.BCEWithLogitsLoss()
    if loss_name == "weighted_bce":
        if train_targets is None: raise ValueError("train_targets requis pour weighted_bce")
        pos = train_targets.sum(axis=0)
        neg = len(train_targets) - pos
        pos_weight = torch.tensor(neg / np.maximum(pos, 1), dtype=torch.float32, device=DEVICE)
        return nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    raise ValueError("Pour la Focal Loss, ajouter/valider une implémentation dédiée avant comparaison.")

## 9.Training loop and evaluation

In [ ]:
def fit_experiment(name, backbone="resnet18", pretrained=True, freeze_backbone=False, loss_name="bce", lr=1e-4,
                   epochs=EPOCHS):
    model = build_model(backbone, pretrained, freeze_backbone)

    # Calcul des cibles train pour la loss pondérée, si sélectionnée.
    train_targets = []
    if loss_name == "weighted_bce":
        #todo : vérifier si on peut faire mieux que ça pour récupérer les targets du train_ds
        #for _, y in DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0):
        #   train_targets.append(y.numpy())
        #train_targets = np.concatenate(train_targets)
        train_targets = np.asarray(train_ds.targets)

    criterion = make_criterion(loss_name, train_targets)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=1, factor=0.5)
    best_loss, best_state, stale = float("inf"), None, 0
    history = []
    start = time.time()

    for epoch in range(epochs):
        model.train()
        running, seen = 0.0, 0
        for images, targets in tqdm(train_loader, desc=f"{name} — époque {epoch + 1}/{epochs}", leave=False):
            images = images.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()
            running += loss.item() * images.size(0)
            seen += images.size(0)

        y_val, p_val = predict_loader(model, val_loader)
        val_loss = float(np.mean([
            criterion(model(images.to(DEVICE)), targets.to(DEVICE)).item()
            for images, targets in []  # loss calculée proprement ci-dessous
        ])) if False else None
        # Évaluation de la loss de validation par batch.
        model.eval()
        total_loss, total_n = 0.0, 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(DEVICE), targets.to(DEVICE)
                batch_loss = criterion(model(images), targets)
                total_loss += batch_loss.item() * images.size(0)
                total_n += images.size(0)
        val_loss = total_loss / max(total_n, 1)
        metrics = compute_metrics(y_val, p_val, threshold=THRESHOLD)
        scheduler.step(val_loss)
        row = {"experiment": name, "epoch": epoch + 1, "train_loss": running / max(seen, 1),
               "val_loss": val_loss, **metrics, "lr": optimizer.param_groups[0]["lr"]}
        history.append(row)
        print({k: round(v, 4) if isinstance(v, (float, np.floating)) else v for k, v in row.items()})
        if val_loss < best_loss:
            best_loss, stale = val_loss, 0
            best_state = copy.deepcopy(model.state_dict())
            torch.save({"model_state_dict": best_state, "backbone": backbone,
                        "pretrained": pretrained, "freeze_backbone": freeze_backbone,
                        "loss_name": loss_name, "threshold": THRESHOLD, "classes": list(classes)},
                       OUTPUT_DIR / f"{name}_best.pt")
        else:
            stale += 1
            if stale >= PATIENCE:
                print("Arrêt anticipé : la loss de validation ne s'améliore plus.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    y_val, p_val = predict_loader(model, val_loader)
    summary = {"experiment": name, "backbone": backbone, "pretrained": pretrained,
               "freeze_backbone": freeze_backbone, "loss": loss_name, "lr": lr,
               "epochs_run": len(history), "best_val_loss": best_loss,
               **compute_metrics(y_val, p_val, THRESHOLD),
               "inverse_frequency_weighted_f1": inverse_frequency_weighted_f1(y_val, (p_val >= THRESHOLD).astype(np.uint8)),
               "seconds": time.time() - start}
    return model, pd.DataFrame(history), summary, y_val, p_val

## 10. Experiments

In [ ]:
EXPERIMENTS = [
    {"name": "resnet18_bce", "backbone": "resnet18", "freeze_backbone": False, "loss_name": "bce", "lr": 1e-4},
    {"name": "resnet18_frozen", "backbone": "resnet18", "freeze_backbone": True, "loss_name": "bce", "lr": 5e-4},
    {"name": "resnet50_bce", "backbone": "resnet50", "freeze_backbone": False, "loss_name": "bce", "lr": 1e-4},
    {"name": "efficientnet_b0_bce", "backbone": "efficientnet_b0", "freeze_backbone": False, "loss_name": "bce",
     "lr": 1e-4},
    {"name": "resnet18_weighted_bce", "backbone": "resnet18", "freeze_backbone": False, "loss_name": "weighted_bce",
     "lr": 1e-4},
]

# Pour un premier test du pipeline, décommenter :
EXPERIMENTS = EXPERIMENTS[:1]

experiment_results = []
histories = {}
prediction_cache = {}
models_cache = {}

for cfg in EXPERIMENTS:
    model, hist, summary, y_val, p_val = fit_experiment(**cfg, epochs=EPOCHS)
    experiment_results.append(summary)
    histories[cfg["name"]] = hist
    prediction_cache[cfg["name"]] = (y_val, p_val)
    models_cache[cfg["name"]] = model
    pd.DataFrame(experiment_results).to_csv(OUTPUT_DIR / "experiment_results_partial.csv", index=False)

results_df = pd.DataFrame(experiment_results).sort_values("f1_macro", ascending=False)
display(results_df)
results_df.to_csv(OUTPUT_DIR / "experiment_results.csv", index=False)

## 11. Threshold tuning

The default threshold of 0.5 is rarely optimal for an imbalanced multi-label problem. For each trained model, we evaluate several thresholds on the validation set and keep the one maximizing the leaderboard F1 (weighted by inverse class frequency).

In [ ]:
best_thresholds = {}
best_scores = {}
plt.figure(figsize=(8, 4))
for name, (y_val, p_val) in prediction_cache.items():
    sweep_df, best = threshold_sweep(y_val, p_val)
    best_thresholds[name] = best["threshold"]
    best_scores[name] = best["f1_leaderboard"]
    print(f"{name} : best threshold = {best['threshold']:.2f} -> leaderboard F1 = {best['f1_leaderboard']:.4f}")
    plt.plot(sweep_df["threshold"], sweep_df["f1_leaderboard"], marker="o", label=name)

plt.axvline(THRESHOLD, color="grey", linestyle="--", label=f"THRESHOLD = {THRESHOLD}")
plt.title("Leaderboard F1 on validation set vs threshold")
plt.xlabel("Threshold")
plt.ylabel("Leaderboard F1")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Test submission

Finally, we have to produce the JSON file in order to submit our predictions to the leaderboard. So we take the model which has the best leaderboard F1 on the validation set, with its own tuned threshold.
```

In [ ]:
test_transform = val_transform if TRANSFORM else resize_transform
test_dataset = COCOTestImageDataset(TEST_IMAGE_DIR, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=PIN_MEMORY)

submission_name = max(best_scores, key=best_scores.get)
submission_model = models_cache[submission_name]
submission_threshold = best_thresholds[submission_name]
print(f"Model : {submission_name} | threshold = {submission_threshold:.2f} | "
      f"validation leaderboard F1 = {best_scores[submission_name]:.4f}")

submission_model.eval()
output_dict = {}
with torch.no_grad():
    for images, names in tqdm(test_loader, desc="Test predictions"):
        probs = torch.sigmoid(submission_model(images.to(DEVICE, non_blocking=True))).cpu()
        for name, prob in zip(names, probs):
            predicted = (prob >= submission_threshold).nonzero().flatten().tolist()
            output_dict[name] = predicted if predicted else [int(prob.argmax())]

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(output_dict, f, indent=4)
